# Modelado Avanzado y Comparativas (Baselines)

**Objetivo:** Entrenar y evaluar una variedad de modelos de machine learning para establecer un rendimiento base (baseline) y comparar modelos más avanzados. Este notebook prueba algoritmos como Regresión Logística, Random Forest y XGBoost.

**Entrada:**
- `data/processed/X_train.csv`, `y_train.csv`
- `data/processed/X_val.csv`, `y_val.csv`
- `data/processed/X_test.csv`, `y_test.csv`

**Salida:**
- **Logs de Consola**: Métricas de rendimiento (AUC, F1-Score, RMSE, MAE) para cada modelo y horizonte de predicción en los conjuntos de validación y prueba.
- **Conocimiento Adquirido**: Identificación de los modelos "campeones" que demuestran el mejor rendimiento y que serán objeto de una optimización más profunda en notebooks posteriores.

# 03 - Advanced Modeling and Baselines

Comparativa avanzada de modelos ML vs baselines naive para el caso espacial.

In [40]:
from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

base_dir = Path.cwd()
for root in [base_dir, *base_dir.parents]:
    if (root / "src").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from src.data_processing.build_dataset import get_training_features

warnings.filterwarnings("ignore")

## 1. Carga de datos y blindaje
Carga los splits, agrega lag espacial y prepara matrices de features/targets.

In [49]:
dataset_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "data" / "processed" / "dataset_entrenamiento_final.csv"
    if candidate.exists():
        dataset_path = candidate
        break
if dataset_path is None:
    raise FileNotFoundError("dataset_entrenamiento_final.csv not found under data/processed")

df = pd.read_csv(dataset_path, parse_dates=["date"])
print("dataset:", dataset_path.resolve())
print("shape:", df.shape)

target_col = "precio_provincial_TARGET_H1"
if target_col not in df.columns:
    raise ValueError(f"Missing target column: {target_col}")

identifiers = ["date", "provincia", "cereal_predominante"]
training_cols = get_training_features(df)
feature_cols = [
    c for c in training_cols
    if c in df.columns and c not in identifiers + [target_col]
]

blacklist = [
    "precio_provincial_lag_1",
    "precio_provincial_lag_2",
    "precio_provincial_lag_3",
    "precio_vecinos_media_lag1",
    "precio_nacional_base_ma3",
    "precio_nacional_base_ma6",
    "precio_nacional_base_vol3",
    "precio_nacional_base_vol6",
]
feature_cols = [c for c in feature_cols if c not in blacklist]
print("Removed blacklist features:", [c for c in blacklist if c in df.columns])

X_full = df[feature_cols].copy()
bool_cols = X_full.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_full[bool_cols] = X_full[bool_cols].astype(int)

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

split_date = pd.Timestamp("2021-01-01")
train_mask = df["date"] < split_date
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

X_train = X_full.loc[train_mask].copy()
X_test = X_full.loc[test_mask].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df[target_col]
y_test = test_df[target_col]

base_price_col = "precio_provincial_lag_1" if "precio_provincial_lag_1" in df.columns else None

required_tokens = {
    "urea": "Urea",
    "dap": "DAP",
    "wheat_intl": "Wheat_Intl",
    "z_clima_adverso": "Clima",
}
feature_names_lower = [c.lower() for c in X_train.columns]
missing_required = [
    label for token, label in required_tokens.items()
    if not any(token in col for col in feature_names_lower)
 ]
if missing_required:
    raise ValueError(f"Missing required feature groups: {missing_required}")

print("target:", target_col)
print("feature count:", len(feature_cols))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("base_price_col:", base_price_col)

dataset: C:\Users\marco\Desktop\Repos\DATAGIA-21\data\processed\dataset_entrenamiento_final.csv
shape: (7047, 76)
Removed blacklist features: ['precio_provincial_lag_1', 'precio_provincial_lag_2', 'precio_provincial_lag_3', 'precio_vecinos_media_lag1', 'precio_nacional_base_ma3', 'precio_nacional_base_ma6', 'precio_nacional_base_vol3', 'precio_nacional_base_vol6']
target: precio_provincial_TARGET_H1
feature count: 64
X_train shape: (5394, 69)
X_test shape: (1653, 69)
base_price_col: precio_provincial_lag_1


## 2. Naive baselines (solo h1)
Baselines con retorno constante para medir el piso de rendimiento.

In [50]:
def directional_accuracy_from_base(y_true, y_pred, base_series):
    return (np.sign(y_pred - base_series) == np.sign(y_true - base_series)).mean()

def compute_metrics(y_true, y_pred, base_series=None):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    pearson = pearsonr(y_true, y_pred)[0] if y_true.nunique() > 1 else np.nan
    if base_series is not None:
        dir_acc = directional_accuracy_from_base(y_true, y_pred, base_series)
    else:
        dir_acc = (np.sign(y_pred) == np.sign(y_true)).mean()
    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "Pearson": float(pearson) if pearson == pearson else np.nan,
        "DirectionalAcc": float(dir_acc) if dir_acc == dir_acc else np.nan,
    }

token_hits = {
    "urea": [c for c in X_train.columns if "urea" in c.lower()][:5],
    "dap": [c for c in X_train.columns if "dap" in c.lower()][:5],
    "wheat_intl": [c for c in X_train.columns if "wheat_intl" in c.lower()][:5],
}
print("Feature sanity check:")
for key, hits in token_hits.items():
    print(f"- {key}: {hits}")

Feature sanity check:
- urea: ['prepag1_urea 46_lag_1', 'prepag1_urea 46_lag_2']
- dap: ['prepag1_dap_lag_1', 'prepag1_dap_lag_2']
- wheat_intl: ['wheat_intl_eur_lag_1', 'wheat_intl_eur_lag_2', 'wheat_intl_eur_lag_3', 'wheat_intl_eur_ma3', 'wheat_intl_eur_vol3']


## 3. Bateria de modelos y tuning (h1, h2, h3)
Busqueda aleatoria con validacion temporal (2019).

In [51]:
forbidden = {
    "precio_provincial_lag_1",
    "precio_provincial_lag_2",
    "precio_provincial_lag_3",
    "precio_vecinos_media_lag1",
    "precio_nacional_base_ma3",
    "precio_nacional_base_ma6",
    "precio_nacional_base_vol3",
    "precio_nacional_base_vol6",
}
present_forbidden = [c for c in X_train.columns if c in forbidden]
print("Forbidden features present in X_train:", present_forbidden)
print("X_train columns:", X_train.shape[1])

Forbidden features present in X_train: []
X_train columns: 69


## 3b. Torneo sin lags de precio
Se excluyen lags de precio provincial para medir senal alfa basada en fundamentales.

In [52]:
tscv = TimeSeriesSplit(n_splits=5)

base_prices_train = train_df[base_price_col] if base_price_col else None
base_prices_test = test_df[base_price_col] if base_price_col else None

def da_scorer(estimator, X, y):
    preds = estimator.predict(X)
    if base_prices_train is None:
        if y.nunique() > 1:
            return pearsonr(y, preds)[0]
        return 0.0
    base = base_prices_train.loc[X.index]
    return directional_accuracy_from_base(y, preds, base)

model_registry = {}
metrics_rows = []
preds_by_model = {}

# Random Forest (n_estimators fixed, tune depth)
rf_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)),
])
rf_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions={"model__max_depth": [4, 6, 8, 12, None]},
    n_iter=5,
    scoring=da_scorer,
    cv=tscv,
    random_state=42,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
rf_best = rf_search.best_estimator_
model_registry["RandomForest"] = rf_best

# XGBoost (use best_params_spatial.json)
config_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "config" / "best_params_spatial.json"
    if candidate.exists():
        config_path = candidate
        break
best_params = {}
if config_path is not None:
    with config_path.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    best_params = payload.get("h1", {}).get("reg", {})

allowed_params = {
    "learning_rate",
    "max_depth",
    "subsample",
    "colsample_bytree",
    "n_estimators",
    "min_child_weight",
    "gamma",
    "reg_lambda",
    "reg_alpha",
}
xgb_params = {k: v for k, v in best_params.items() if k in allowed_params}
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    **xgb_params,
 )
xgb_model.fit(X_train, y_train)
model_registry["XGBoost"] = xgb_model

# Evaluate all models
for name, model in model_registry.items():
    preds = model.predict(X_test)
    metrics = compute_metrics(y_test, preds, base_prices_test)
    metrics_rows.append({"model": name, **metrics})
    preds_by_model[name] = preds

comparison_df = pd.DataFrame(metrics_rows).sort_values(
    ["DirectionalAcc", "Pearson"], ascending=False
 )
comparison_df

,model,MAE,RMSE,Pearson,DirectionalAcc
1,XGBoost,6.500466,7.981585,0.540974,0.618875
0,RandomForest,5.818722,7.383514,0.607867,0.600726


In [53]:
winner_row = comparison_df.iloc[0]
print("Campeon DATAGIA:")
print(winner_row)

Campeon DATAGIA:
model              XGBoost
MAE               6.500466
RMSE              7.981585
Pearson           0.540974
DirectionalAcc    0.618875
Name: 1, dtype: object


## 4. Evaluacion granular del mejor modelo por horizonte
Metricas desglosadas por provincia y cereal.

In [54]:
champion_name = comparison_df.iloc[0]["model"]
champion_preds = preds_by_model[champion_name]

residual_df = test_df[["date", "provincia", "cereal_predominante"]].copy()
residual_df["y_true"] = y_test.values
residual_df["y_pred"] = champion_preds
residual_df["residual"] = residual_df["y_pred"] - residual_df["y_true"]
residual_df["abs_error"] = residual_df["residual"].abs()

crisis_df = residual_df[residual_df["date"].dt.year == 2022].copy()
prov_errors = (
    crisis_df.groupby("provincia")["abs_error"].mean()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
 )
month_errors = (
    crisis_df.groupby(crisis_df["date"].dt.to_period("M"))
    ["abs_error"].mean()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
 )
month_errors["date"] = month_errors["date"].astype(str)

print("Top provincias con mayor error medio (2022):")
print(prov_errors)
print("\nTop meses con mayor error medio (2022):")
print(month_errors)

Top provincias con mayor error medio (2022):
     provincia  abs_error
0       Cuenca  15.565236
1  Ciudad Real  15.054135
2     Albacete  14.861983
3       Toledo  14.754478
4       Huesca  14.491980

Top meses con mayor error medio (2022):
      date  abs_error
0  2022-04  16.803740
1  2022-05  16.560332
2  2022-09  15.270950
3  2022-06  15.055814
4  2022-07  14.424845


## 5. Simulador de arbitraje (h1, regresion)
Estimacion de compra/venta por cereal usando el mejor modelo de regresion.

In [55]:
def get_feature_importances(model, feature_names):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        order = np.argsort(importances)[::-1]
        return [(feature_names[i], float(importances[i])) for i in order]
    return []

rf_model = model_registry.get("RandomForest")
xgb_model = model_registry.get("XGBoost")

rf_importances = get_feature_importances(
    rf_model.named_steps["model"] if hasattr(rf_model, "named_steps") else rf_model,
    X_train.columns.tolist(),
)
xgb_importances = get_feature_importances(xgb_model, X_train.columns.tolist())

def rank_feature(importances, token):
    for idx, (feat, _) in enumerate(importances, start=1):
        if token in feat.lower():
            return idx
    return None

rf_urea_rank = rank_feature(rf_importances, "urea")
xgb_urea_rank = rank_feature(xgb_importances, "urea")
rf_top10 = rf_importances[:10]
xgb_top10 = xgb_importances[:10]

rf_top5 = [feat for feat, _ in rf_importances[:5]]
xgb_top5 = [feat for feat, _ in xgb_importances[:5]]

print("RF Top10:", rf_top10)
print("XGB Top10:", xgb_top10)
print("Urea rank RF:", rf_urea_rank)
print("Urea rank XGB:", xgb_urea_rank)
print("Urea in RF Top5:", any("urea" in f.lower() for f in rf_top5))
print("Urea in XGB Top5:", any("urea" in f.lower() for f in xgb_top5))
print("Wheat intl in RF Top5:", any("wheat_intl" in f.lower() for f in rf_top5))
print("Wheat intl in XGB Top5:", any("wheat_intl" in f.lower() for f in xgb_top5))
print("Clima in RF Top5:", any("z_clima_adverso" in f.lower() for f in rf_top5))
print("Clima in XGB Top5:", any("z_clima_adverso" in f.lower() for f in xgb_top5))

RF Top10: [('prepag2_torta de girasol_lag_1', 0.5626799600823041), ('wheat_intl_eur_ma3', 0.23919439012018837), ('corn_intl_eur_ma3', 0.14428210117607063), ('sup_cereal_predominante_lag_1', 0.011247053569229488), ('idx_semillas_lag_1', 0.006393813742377821), ('month_sin', 0.005944574393397005), ('corn_intl_eur_lag_1', 0.005619889910641957), ('total_sup_trigo_lag_1', 0.0025786718053864306), ('idx_gastos_generales_lag_1', 0.0025342475613374395), ('prepag1_urea 46_lag_1', 0.002113471038950238)]
XGB Top10: [('prepag2_torta de girasol_lag_1', 0.2392968386411667), ('wheat_intl_eur_ma3', 0.18939988315105438), ('wheat_intl_eur_lag_1', 0.16833727061748505), ('corn_intl_eur_ma3', 0.06554654240608215), ('corn_intl_eur_lag_1', 0.058654241263866425), ('prepag2_torta de girasol_lag_2', 0.03609005734324455), ('idx_fitosanitarios_lag_1', 0.014026772230863571), ('idx_fertilizantes_lag_2', 0.013671409338712692), ('lat_centroide', 0.012202552519738674), ('idx_bienes_inversion_lag_1', 0.011875857599079609

## 6. Resumen para reporte de saneamiento
Se selecciona el mejor modelo global y se guarda un resumen para el reporte final.

In [56]:
baseline_da_gate = 0.58

podio = comparison_df.copy()
champion = podio.iloc[0]
apt_prod = champion["DirectionalAcc"] > baseline_da_gate
apt_text = "Apto para Produccion" if apt_prod else "No Apto para Produccion"

lines = []
lines.append("# REPORTE MODELO CAMPEON - Caso 21")
lines.append("")
lines.append("## Podio de Modelos (H1) - Senal Pura")
lines.append("")
lines.append("| Modelo | Pearson | DA | MAE | RMSE |")
lines.append("| --- | ---: | ---: | ---: | ---: |")
for _, row in podio.iterrows():
    lines.append(
        f"| {row['model']} | {row['Pearson']:.4f} | {row['DirectionalAcc']:.4f} | {row['MAE']:.4f} | {row['RMSE']:.4f} |"
    )

lines.append("")
lines.append("## Analisis de residuos (crisis 2022)")
lines.append("")
lines.append("Top provincias con mayor error medio:")
for _, row in prov_errors.iterrows():
    lines.append(f"- {row['provincia']}: MAE={row['abs_error']:.4f}")
lines.append("")
lines.append("Top meses con mayor error medio:")
for _, row in month_errors.iterrows():
    lines.append(f"- {row['date']}: MAE={row['abs_error']:.4f}")

lines.append("")
lines.append("## Importancia cruzada (Urea) - RF vs XGB")
lines.append("")
lines.append(f"- RF Urea rank: {rf_urea_rank}")
lines.append(f"- XGB Urea rank: {xgb_urea_rank}")

lines.append("")
lines.append("## Prueba de Resiliencia: Senal Agronomica y Macro Pura")
lines.append("")
lines.append("Se excluyeron variables de nivel de precio y volatilidad: precio_provincial_lag_1/2/3, precio_vecinos_media_lag1, precio_nacional_base_ma3/ma6, precio_nacional_base_vol3/vol6.")
lines.append("")
lines.append("Indicadores Top5 en importancia:")
lines.append(f"- Urea en RF Top5: {any('urea' in f.lower() for f in rf_top5)}")
lines.append(f"- Urea en XGB Top5: {any('urea' in f.lower() for f in xgb_top5)}")
lines.append(f"- Wheat intl en RF Top5: {any('wheat_intl' in f.lower() for f in rf_top5)}")
lines.append(f"- Wheat intl en XGB Top5: {any('wheat_intl' in f.lower() for f in xgb_top5)}")
lines.append(f"- Clima en RF Top5: {any('z_clima_adverso' in f.lower() for f in rf_top5)}")
lines.append(f"- Clima en XGB Top5: {any('z_clima_adverso' in f.lower() for f in xgb_top5)}")

lines.append("")
lines.append("## Certificacion de produccion")
lines.append("")
lines.append(f"- Campeon: {champion['model']}")
lines.append(f"- {apt_text} (criterio: DA > {baseline_da_gate:.2f})")

report_root = None
for root in [base_dir, *base_dir.parents]:
    if (root / "data").exists():
        report_root = root
        break
if report_root is None:
    report_root = base_dir

report_dir = report_root / "reports"
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "REPORTE_MODELO_CAMPEON.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved report:", report_path.resolve())

Saved report: C:\Users\marco\Desktop\Repos\DATAGIA-21\reports\REPORTE_MODELO_CAMPEON.md
